<a href="https://colab.research.google.com/github/facundosouza3-rgb/Souza_Facundo_Preentrega-opcional---Limpieza/blob/main/Preentrega_opcional_Limpieza.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pre-entrega: Limpieza y análisis exploratorio de ventas de una cafetería

**Fuente:** Kaggle, "Cafe Sales - Dirty Data for Cleaning Training" (dirty_cafe_sales.csv)

**Pregunta de investigación:** ¿Qué productos, métodos de pago y ubicaciones generan más ingresos, y cómo evolucionan las ventas en el tiempo?


In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("dirty_cafe_sales.csv")
df.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [2]:
print("Filas y columnas:", df.shape)
df.info()

Filas y columnas: (10000, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    10000 non-null  object
 1   Item              9667 non-null   object
 2   Quantity          9862 non-null   object
 3   Price Per Unit    9821 non-null   object
 4   Total Spent       9827 non-null   object
 5   Payment Method    7421 non-null   object
 6   Location          6735 non-null   object
 7   Transaction Date  9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB


In [3]:
print("Nulos por columna:")
print(df.isnull().sum())
print("\nDuplicados exactos:", df.duplicated().sum())
print("Transaction ID duplicados:", df["Transaction ID"].duplicated().sum())

Nulos por columna:
Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64

Duplicados exactos: 0
Transaction ID duplicados: 0


In [4]:
basura = df.isin(["ERROR", "UNKNOWN"]).sum()
print("Valores 'ERROR' o 'UNKNOWN' por columna:")
print(basura)

Valores 'ERROR' o 'UNKNOWN' por columna:
Transaction ID        0
Item                636
Quantity            341
Price Per Unit      354
Total Spent         329
Payment Method      599
Location            696
Transaction Date    301
dtype: int64


## Diagnóstico inicial

- El dataset tiene 10.000 filas y 8 columnas.
- Las 8 columnas se leen como texto (str), incluso Quantity, Price Per Unit, Total Spent y Transaction Date, que deberían ser numéricas y de fecha.
- Hay nulos reales, sobre todo en Payment Method y Location.
- Además hay valores "ERROR" y "UNKNOWN" que Pandas no cuenta como nulos y que impiden convertir los tipos.
- No hay duplicados: Transaction ID es único en todas las filas.

In [5]:
# Unificar "ERROR" y "UNKNOWN" como nulos reales
df = df.replace(["ERROR", "UNKNOWN"], np.nan)

# Convertir a los tipos correctos
for col in ["Quantity", "Price Per Unit", "Total Spent"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["Transaction Date"] = pd.to_datetime(df["Transaction Date"], errors="coerce")

df.info()
print(df.isnull().sum())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID    10000 non-null  object        
 1   Item              9031 non-null   object        
 2   Quantity          9521 non-null   float64       
 3   Price Per Unit    9467 non-null   float64       
 4   Total Spent       9498 non-null   float64       
 5   Payment Method    6822 non-null   object        
 6   Location          6039 non-null   object        
 7   Transaction Date  9540 non-null   datetime64[ns]
dtypes: datetime64[ns](1), float64(3), object(4)
memory usage: 625.1+ KB
Transaction ID         0
Item                 969
Quantity             479
Price Per Unit       533
Total Spent          502
Payment Method      3178
Location            3961
Transaction Date     460
dtype: int64


In [6]:
df.describe()

,Quantity,Price Per Unit,Total Spent,Transaction Date
count,9521.000000,9467.000000,9498.000000,9540
mean,3.028463,2.949984,8.924352,2023-07-01 23:00:31.698113280
min,1.000000,1.000000,1.000000,2023-01-01 00:00:00
25%,2.000000,2.000000,4.000000,2023-04-01 00:00:00
50%,3.000000,3.000000,8.000000,2023-07-02 00:00:00
75%,4.000000,4.000000,12.000000,2023-10-02 00:00:00
max,5.000000,5.000000,25.000000,2023-12-31 00:00:00
std,1.419007,1.278450,6.009919,NaN


## Resumen numérico

- Quantity y Price Per Unit toman valores entre 1 y 5; Total Spent entre 1 y 25, que es exactamente 5 x 5, por lo que los rangos son consistentes entre sí.
- Transaction Date cubre todo el año 2023, sin fechas fuera de rango.
- No hay valores negativos ni extremos: el problema del dataset no son los outliers sino los nulos y los valores "ERROR"/"UNKNOWN".
- Tras unificar "ERROR" y "UNKNOWN" como nulos, las columnas con más faltantes son Location (3.961) y Payment Method (3.178).

In [7]:
# Total Spent = Quantity * Price Per Unit
m = df["Total Spent"].isna() & df["Quantity"].notna() & df["Price Per Unit"].notna()
df.loc[m, "Total Spent"] = df["Quantity"] * df["Price Per Unit"]

# Price Per Unit = Total Spent / Quantity
m = df["Price Per Unit"].isna() & df["Total Spent"].notna() & df["Quantity"].notna()
df.loc[m, "Price Per Unit"] = df["Total Spent"] / df["Quantity"]

# Quantity = Total Spent / Price Per Unit
m = df["Quantity"].isna() & df["Total Spent"].notna() & df["Price Per Unit"].notna()
df.loc[m, "Quantity"] = df["Total Spent"] / df["Price Per Unit"]

print(df[["Quantity", "Price Per Unit", "Total Spent"]].isnull().sum())

Quantity          38
Price Per Unit    38
Total Spent       40
dtype: int64


In [8]:
# Cada Item tiene un único precio: lo tomamos de las filas donde sí está
precios = df.dropna(subset=["Item", "Price Per Unit"]).groupby("Item")["Price Per Unit"].agg(lambda s: s.mode()[0])
print(precios)

m = df["Price Per Unit"].isna() & df["Item"].notna()
df.loc[m, "Price Per Unit"] = df.loc[m, "Item"].map(precios)

# Repetir las reconstrucciones aritméticas con el precio ya completado
m = df["Total Spent"].isna() & df["Quantity"].notna() & df["Price Per Unit"].notna()
df.loc[m, "Total Spent"] = df["Quantity"] * df["Price Per Unit"]

m = df["Quantity"].isna() & df["Total Spent"].notna() & df["Price Per Unit"].notna()
df.loc[m, "Quantity"] = df["Total Spent"] / df["Price Per Unit"]

print(df[["Quantity", "Price Per Unit", "Total Spent"]].isnull().sum())

Item
Cake        3.0
Coffee      2.0
Cookie      1.0
Juice       3.0
Salad       5.0
Sandwich    4.0
Smoothie    4.0
Tea         1.5
Name: Price Per Unit, dtype: float64
Quantity          23
Price Per Unit     6
Total Spent       23
dtype: int64


In [9]:
antes = len(df)
df = df.dropna(subset=["Quantity", "Price Per Unit", "Total Spent"])
df = df.reset_index(drop=True)
print("Filas eliminadas:", antes - len(df))
print("Filas restantes:", len(df))

Filas eliminadas: 26
Filas restantes: 9974


In [10]:
for col in ["Item", "Payment Method", "Location"]:
    df[col] = df[col].fillna("Sin dato")

df.isnull().sum()

,0
Transaction ID,0
Item,0
Quantity,0
Price Per Unit,0
Total Spent,0
Payment Method,0
Location,0
Transaction Date,460


## Saneamiento: filtros booleanos

- Filtro 1: se descartan las filas sin producto identificado (Item = "Sin dato"), porque sin producto no se puede responder la pregunta de investigación.
- Filtro 2: se descartan las filas sin fecha de transacción, para poder analizar la evolución mensual.
- Filtro 3: se crea un subconjunto (df_completo) con las transacciones que tienen método de pago y ubicación conocidos, para analizar esas dos variables sin mezclar categorías "Sin dato".
- Se elimina la columna Transaction ID: es un identificador único por fila y no aporta al análisis.

In [11]:
print("Filas antes:", len(df))

# Filtro 1: producto identificado
df = df[df["Item"] != "Sin dato"]
print("Tras filtro 1 (Item conocido):", len(df))

# Filtro 2: fecha válida
df = df[df["Transaction Date"].notna()]
print("Tras filtro 2 (fecha válida):", len(df))

# Eliminar columna innecesaria
df = df.drop(columns=["Transaction ID"])

# Resetear el índice tras eliminar filas
df = df.reset_index(drop=True)

# Filtro 3: subconjunto con pago y ubicación conocidos
df_completo = df[(df["Payment Method"] != "Sin dato") & (df["Location"] != "Sin dato")]
print("Filtro 3 (pago y ubicación conocidos):", len(df_completo))

print("\nColumnas finales:", list(df.columns))
df.head()

Filas antes: 9974
Tras filtro 1 (Item conocido): 9011
Tras filtro 2 (fecha válida): 8593
Filtro 3 (pago y ubicación conocidos): 3570

Columnas finales: ['Item', 'Quantity', 'Price Per Unit', 'Total Spent', 'Payment Method', 'Location', 'Transaction Date']


,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,Cake,4.0,3.0,12.0,Cash,In-store,2023-05-16
2,Cookie,4.0,1.0,4.0,Credit Card,In-store,2023-07-19
3,Salad,2.0,5.0,10.0,Sin dato,Sin dato,2023-04-27
4,Coffee,2.0,2.0,4.0,Digital Wallet,In-store,2023-06-11


Group by
1. Ingresos por producto
2. Ingresos por método de pago y ubicación
3. Evolución mensual

In [12]:
por_producto = (
    df.groupby("Item")["Total Spent"]
    .agg(transacciones="count", ingreso_total="sum", ticket_promedio="mean")
    .sort_values("ingreso_total", ascending=False)
    .round(2)
    .reset_index()
)
por_producto

,Item,transacciones,ingreso_total,ticket_promedio
0,Salad,1097,16575.0,15.11
1,Sandwich,1071,13068.0,12.20
2,Smoothie,1046,12776.0,12.21
3,Juice,1121,10023.0,8.94
4,Cake,1081,9834.0,9.10
5,Coffee,1118,6814.0,6.09
6,Tea,1026,4665.0,4.55
7,Cookie,1033,3076.0,2.98


La cantidad de transacciones es muy parecida entre productos (entre 1.026 y 1.121), por lo que las diferencias de ingreso vienen del precio y no del volumen. Salad (16.575) es el producto que más factura y Cookie (3.076) el que menos. Salad, Sandwich y Smoothie suman cerca del 55 % del ingreso total.

In [13]:
por_pago_ubicacion = (
    df_completo.groupby(["Payment Method", "Location"])["Total Spent"]
    .agg(transacciones="count", ingreso_total="sum")
    .round(2)
    .reset_index()
)
por_pago_ubicacion

,Payment Method,Location,transacciones,ingreso_total
0,Cash,In-store,601,5522.0
1,Cash,Takeaway,566,5052.0
2,Credit Card,In-store,581,5175.5
3,Credit Card,Takeaway,582,5158.0
4,Digital Wallet,In-store,595,5663.0
5,Digital Wallet,Takeaway,645,5475.5


El ingreso está repartido de forma bastante pareja entre las seis combinaciones (entre 5.052 y 5.663). La más alta es Digital Wallet en el local (In-store) y la más baja es Cash para llevar (Takeaway), con una diferencia de alrededor del 12 %. No hay un método de pago ni una ubicación dominante. Este análisis usa solo las 3.570 filas con ambos datos conocidos, por lo que hay que tomarlo con cautela.

In [14]:
por_mes = (
    df.groupby(df["Transaction Date"].dt.month)["Total Spent"]
    .agg(transacciones="count", ingreso_total="sum")
    .round(2)
    .reset_index()
    .rename(columns={"Transaction Date": "mes"})
)
por_mes

,mes,transacciones,ingreso_total
0,1,742,6690.0
1,2,649,5953.5
2,3,743,6526.5
3,4,696,6464.5
4,5,695,6220.0
5,6,733,6616.0
6,7,717,6175.5
7,8,715,6348.0
8,9,706,6208.0
9,10,767,6755.0


El ingreso mensual es estable, entre 5.953 y 6.755. Febrero es el mes más bajo (5.953,5) y octubre el más alto (6.755, con 767 transacciones). No se observa una estacionalidad marcada en 2023.

## Reflexión final

**Problemas encontrados**
- Las 8 columnas se leían como texto, incluso las numéricas y la fecha. Hubo que convertir Quantity, Price Per Unit, Total Spent y Transaction Date.
- Había valores "ERROR" y "UNKNOWN" que Pandas no reconoce como nulos; se unificaron como nulos reales.
- Había muchos nulos, sobre todo en Location (3.961) y Payment Method (3.178). Para no perder más de un tercio del dataset no se eliminaron esas filas; se marcaron como "Sin dato".
- Los faltantes numéricos se recuperaron con la relación Total Spent = Quantity x Price Per Unit y con el precio fijo de cada producto. Solo se eliminaron 26 filas que no se podían reconstruir.
- No hay duplicados: Transaction ID es único en todas las filas.
- Se perdieron 1.407 filas en total (14 %) por los filtros de Item y de fecha, quedando 8.593.

**Variables clave**
- Item, Quantity, Price Per Unit y Total Spent, para el análisis de ingresos.
- Transaction Date (mes), para la evolución en el tiempo.
- Payment Method y Location, aunque con la limitación de que solo el 41 % de las filas tiene ambos datos conocidos.



*
**

FIN




